In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import os

os.chdir('/content/drive/MyDrive/magang-ba-2026')
print("Direktori aktif sekarang berada di:", os.getcwd())

Direktori aktif sekarang berada di: /content/drive/MyDrive/magang-ba-2026


**Menyiapkan Basis Data Uji Baru**

In [32]:
import os
import glob
import shutil
import sqlite3

# 1. Pastikan folder bukti tersedia
os.makedirs('reports/evidence', exist_ok=True)
os.makedirs('data/backup', exist_ok=True)

# 2. Cari file database utama di proyek
db_files = [f for f in (glob.glob('**/*.db', recursive=True) + glob.glob('**/*.sqlite*', recursive=True)) if 'backup' not in f]
print("File database ditemukan:", db_files)

# 3. Backup otomatis database utama (jika ada) sebelum membuat DB uji baru
for db_path in db_files:
    backup_path = os.path.join('data/backup', f"backup_sebelum_D32_{os.path.basename(db_path)}")
    shutil.copy2(db_path, backup_path)
    print(f"Backup aman dibuat di: {backup_path}")

# 4. Cek struktur docs/bug_list.csv yang sudah ada
if os.path.exists('docs/bug_list.csv'):
    with open('docs/bug_list.csv', 'r', encoding='utf-8') as f:
        print("\nHeader docs/bug_list.csv saat ini:", f.readline().strip())

File database ditemukan: ['data/app.db', 'data/app_test.db', 'data/sentiment_d28.db', 'data/sentiment.db']
Backup aman dibuat di: data/backup/backup_sebelum_D32_app.db
Backup aman dibuat di: data/backup/backup_sebelum_D32_app_test.db
Backup aman dibuat di: data/backup/backup_sebelum_D32_sentiment_d28.db
Backup aman dibuat di: data/backup/backup_sebelum_D32_sentiment.db

Header docs/bug_list.csv saat ini: bug_id,tc_id,modul,tingkat_keparahan,deskripsi_masalah,langkah_reproduksi,status_perbaikan,bukti


**Mengosongkan Database Uji Aktif**

In [33]:
import sqlite3
import os

# Cek isi tabel pada masing-masing database untuk mengetahui mana yang dipakai aplikasi
for db_path in ['data/app.db', 'data/sentiment.db']:
    if os.path.exists(db_path):
        conn = sqlite3.connect(db_path)
        cursor = conn.cursor()
        tables = cursor.execute("SELECT name FROM sqlite_master WHERE type='table';").fetchall()
        print(f"Database {db_path} memiliki tabel:", [t[0] for t in tables])
        for t in tables:
            count = cursor.execute(f"SELECT COUNT(*) FROM {t[0]}").fetchone()[0]
            print(f"   - {t[0]}: {count} baris")
        conn.close()

Database data/app.db memiliki tabel: ['posts', 'comments', 'annotators', 'sqlite_sequence', 'annotations']
   - posts: 120 baris
   - comments: 915 baris
   - annotators: 0 baris
   - sqlite_sequence: 0 baris
   - annotations: 0 baris
Database data/sentiment.db memiliki tabel: ['predictions', 'sqlite_sequence', 'comments', 'verification_history']
   - predictions: 0 baris
   - sqlite_sequence: 0 baris
   - comments: 0 baris
   - verification_history: 0 baris


**Reset Database Uji Baru (Kosongkan Baris, Pertahankan Skema)**

In [34]:
import sqlite3

def reset_db(db_path, tables):
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    # Matikan pengecekan foreign key sementara agar urutan penghapusan aman
    cursor.execute("PRAGMA foreign_keys = OFF;")
    for table in tables:
        cursor.execute(f"DELETE FROM {table};")
        # Reset auto-increment jika tabel ada di sqlite_sequence
        cursor.execute("DELETE FROM sqlite_sequence WHERE name = ?;", (table,))
    cursor.execute("PRAGMA foreign_keys = ON;")
    conn.commit()

    print(f"{db_path} berhasil direset (0 baris):")
    for table in tables:
        cnt = cursor.execute(f"SELECT COUNT(*) FROM {table};").fetchone()[0]
        print(f"   - {table}: {cnt} baris")
    conn.close()

# Reset kedua database aktif
reset_db('data/app.db', ['annotations', 'annotators', 'comments', 'posts'])
reset_db('data/sentiment.db', ['verification_history', 'predictions', 'comments'])

data/app.db berhasil direset (0 baris):
   - annotations: 0 baris
   - annotators: 0 baris
   - comments: 0 baris
   - posts: 0 baris
data/sentiment.db berhasil direset (0 baris):
   - verification_history: 0 baris
   - predictions: 0 baris
   - comments: 0 baris


**Mencatat bug_list**

In [35]:
import os
import pandas as pd
from openpyxl import load_workbook

# 1. Buat docs/bug_list.csv
os.makedirs("docs", exist_ok=True)
bugs = [
    {
        "bug_id": "BUG-01",
        "tc_id": "TC-05",
        "modul": "Impor Data (src/importer.py)",
        "tingkat_keparahan": "Tinggi",
        "deskripsi_masalah": "Baris komentar dengan tanggal tidak valid (2026-99-99 25:61:00) dan di luar rentang (1998-05-12 08:00:00) lolos diimpor (Diterima: 2) tanpa penolakan aturan validasi tanggal Q05/Q06.",
        "langkah_reproduksi": "1. Buka menu Impor -> Comments. 2. Unggah tc05_tanggal_invalid.csv. 3. Klik Mulai Impor Comments.",
        "status_perbaikan": "Terbuka",
        "bukti": "reports/evidence/T05.png"
    },
    {
        "bug_id": "BUG-02",
        "tc_id": "TC-07",
        "modul": "Eksplorasi Data (src/queries.py)",
        "tingkat_keparahan": "Tinggi",
        "deskripsi_masalah": "Kombinasi filter rentang tanggal bersama platform Instagram memicu TypeError: Invalid comparison between dtype=datetime64[s] and date pada src/queries.py baris 158 saat kolom commented_at kosong/NaT.",
        "langkah_reproduksi": "1. Buka menu Data. 2. Pilih Platform Instagram dan isi Dari Tanggal (2026/09/01) serta Sampai Tanggal (2026/09/30).",
        "status_perbaikan": "Terbuka",
        "bukti": "reports/evidence/TC07.png"
    },
    {
        "bug_id": "BUG-03",
        "tc_id": "TC-15",
        "modul": "Ekspor Data (app/pages/6_Ekspor.py)",
        "tingkat_keparahan": "Sedang",
        "deskripsi_masalah": "Halaman Ekspor hanya mengunduh berkas CSV tunggal tanpa menyertakan file manifest/metadata ekspor (versi data, versi model, parameter filter, dan jumlah baris).",
        "langkah_reproduksi": "1. Buka menu Ekspor. 2. Klik Unduh Berkas CSV (Filtered Export). 3. Periksa berkas terunduh dan direktori proyek.",
        "status_perbaikan": "Terbuka",
        "bukti": "reports/evidence/T15.png"
    }
]

df_bugs = pd.DataFrame(bugs)
df_bugs.to_csv("docs/bug_list.csv", index=False, encoding="utf-8")
print("docs/bug_list.csv berhasil dibuat:")
print(df_bugs[["bug_id", "tc_id", "modul", "tingkat_keparahan", "status_perbaikan"]])

docs/bug_list.csv berhasil dibuat:
   bug_id  tc_id                                modul tingkat_keparahan  \
0  BUG-01  TC-05         Impor Data (src/importer.py)            Tinggi   
1  BUG-02  TC-07     Eksplorasi Data (src/queries.py)            Tinggi   
2  BUG-03  TC-15  Ekspor Data (app/pages/6_Ekspor.py)            Sedang   

  status_perbaikan  
0          Terbuka  
1          Terbuka  
2          Terbuka  


Memperbarui test/test_case.xls

In [36]:
# 2. Perbarui tests/test_cases.xlsx
hasil_uat = {
    "TC-01": ("2 baris posts dan 5 baris comments valid berhasil diimpor dan tampil lengkap di halaman Data (Ditemukan 5 Komentar).", "Lulus", "reports/evidence/TC01(4).png"),
    "TC-02": ("File tc02_kolom_hilang.csv ditolak dengan pesan eksplisit: 'File ditolak! Kolom wajib berikut hilang dari file: text_original'.", "Lulus", "reports/evidence/TC02.png"),
    "TC-03": ("File tc03_encoding_rusak.csv ditolak secara bersih dengan pesan: 'File ditolak! Encoding file tidak valid (bukan UTF-8)' tanpa crash.", "Lulus", "reports/evidence/TC03.png"),
    "TC-04": ("Kedua baris dengan post_id tidak terdaftar ditolak dengan alasan 'Q09: unggahan sumber tidak ditemukan' (Diterima: 0, Ditolak: 2).", "Lulus", "reports/evidence/T04(2).png"),
    "TC-05": ("Kedua baris dengan tanggal invalid (2026-99-99 & 1998-05-12) lolos masuk ke database (Diterima: 2, Ditolak: 0) tanpa penolakan Q05/Q06.", "Gagal", "reports/evidence/T05.png"),
    "TC-06": ("Impor ulang file tc01_comments_valid.csv tidak menambah baris duplikat di database (Diterima: 0, Dilewati: 5).", "Lulus", "reports/evidence/T06(2).png"),
    "TC-07": ("Filter rentang tanggal bersama platform Instagram memicu TypeError: Invalid comparison between dtype=datetime64[s] and date pada src/queries.py baris 158.", "Gagal", "reports/evidence/TC07.png"),
    "TC-08": ("Kata kunci 'daftar' menampilkan 2 komentar sesuai; kata kunci tidak ada ('abcd') menampilkan 'Ditemukan 0 Komentar' tanpa galat.", "Lulus", "reports/evidence/TC08(2).png"),
    "TC-09": ("Proses analisis sentimen berhasil memproses 5 komentar lengkap dengan predicted_label, score, predicted_at, dan model_version T3_indobert_len64.", "Lulus", "reports/evidence/T09(2).png"),
    "TC-10": ("Eksekusi ulang analisis pada komentar yang sama menambahkan 5 riwayat prediksi baru (2026-11-18T11:27:23) tanpa menimpa riwayat sebelumnya (11:26:21).", "Lulus", "reports/evidence/T10(2).png"),
    "TC-11": ("Koreksi label TEST_CMT_001 (positif -> netral) tersimpan lengkap beserta reviewer_azza, alasan, waktu, dan mempertahankan prediksi asli model.", "Lulus", "reports/evidence/T11(2).png"),
    "TC-12": ("Pengajuan koreksi dengan alasan kosong ditolak dengan pesan 'Validasi Gagal: Alasan minimal 5 karakter'.", "Lulus", "reports/evidence/T12.png"),
    "TC-13": ("Dashboard menampilkan metrik akurat (5 unggahan, 5 diprediksi, 2 terverifikasi / 40.0%) beserta pemilih sumber label dan catatan cakupan data.", "Lulus", "reports/evidence/T13(1).png"),
    "TC-14": ("Ekspor terfilter menghasilkan file CSV berisi 5 baris dan 21 kolom sesuai kontrak tanpa kolom identitas pribadi.", "Lulus", "reports/evidence/T14.png"),
    "TC-15": ("Halaman Ekspor hanya menyediakan unduhan berkas CSV tanpa menghasilkan file manifest/metadata ekspor (versi model, filter, versi data).", "Gagal", "reports/evidence/T15.png"),
}

wb = load_workbook("tests/test_cases.xlsx")
ws = wb.active
headers = {str(cell.value).strip().lower(): idx + 1 for idx, cell in enumerate(ws[1]) if cell.value}

for row_idx in range(2, ws.max_row + 1):
    tc_id = str(ws.cell(row=row_idx, column=headers.get("id", 1)).value or "").strip()
    if tc_id in hasil_uat:
        aktual, status, bukti = hasil_uat[tc_id]
        for col_name, val in [
            ("hasil aktual", aktual),
            ("status", status),
            ("tanggal uji", "2026-11-18"),
            ("penguji", "Azzahra Attaqina"),
            ("bukti", bukti),
        ]:
            for h_key, col_idx in headers.items():
                if col_name in h_key:
                    ws.cell(row=row_idx, column=col_idx, value=val)

wb.save("tests/test_cases.xlsx")
print("tests/test_cases.xlsx berhasil diperbarui.")

tests/test_cases.xlsx berhasil diperbarui.


**Memulihkan database asli dari backup**

In [4]:
import os
import shutil
from google.colab import drive

drive.mount('/content/drive')

project_dir = '/content/drive/MyDrive/magang-ba-2026'
os.chdir(project_dir)

backup_dir = "data/backup"

for db_name in ["app.db", "sentiment.db"]:
    backup_src = os.path.join(
        backup_dir,
        f"backup_sebelum_D32_{db_name}"
    )
    target_dst = os.path.join("data", db_name)

    if os.path.isfile(backup_src):
        shutil.copy2(backup_src, target_dst)
        print(f"Database lama dipulihkan ke: {target_dst}")
    else:
        print(f"Backup tidak ditemukan: {backup_src}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Database lama dipulihkan ke: data/app.db
Database lama dipulihkan ke: data/sentiment.db
